# 16 · Measuring search quality
Level L3 · Part 3 · builds on lesson 15

Needs: nothing. Everything runs in local mode (`QdrantClient(":memory:")`) on the saved vectors
in `data/`. One `needs-download` cell re-embeds the test queries with MiniLM (about 90 MB, cached
once); when it is skipped the saved query cache is used. Importing `ranx` compiles its metrics
the first time (up to a minute, then about 10 s).

## Goal
By the end of this lesson you can write a labelled set of 20 recipe queries, compute
precision@k, recall@k, MRR and nDCG for the recipe finder by hand and with `ranx`, and use the
numbers to compare two search methods query by query.

## The idea
Lesson 15 checked a few top hits against one word in the title, and the check was easy to fool.
To *measure* search you need an answer key. For each test query, list the recipes a person would
accept, each with a grade. Those are **relevance labels**; the search's ranked list is scored
against them. Each number answers a different question: how much of the page is useful
(precision), how much of the good stuff was found (recall), how soon the first good answer
appears (MRR), and whether the best answers come first (nDCG). Like a teacher's marking scheme,
every new version of the search sits the same exam.

## Picture

```mermaid
flowchart LR
    Q["20 test queries"] --> L["relevance labels<br/>query -> {slug: grade}"]
    Q --> S["search(query, k=10)<br/>ranked slugs"]
    S --> RUN["run<br/>query -> [slug, slug, ...]"]
    L --> M["precision@k, recall@k,<br/>reciprocal rank, nDCG@k"]
    RUN --> M
    M --> T["per-query table<br/>+ mean row"]
    T --> C["compare methods:<br/>vector vs keyword"]
```

## Step by step
### 1. The recipe finder from lesson 15
Load the 2,000 recipes and their MiniLM vectors, and check they line up row by row. This lesson
measures ranking only, so the collection needs no filters or payload indexes: the payload holds
just the `slug` (the recipe's permanent key, lesson 14) and the title.

In [1]:
import json, math, os, re, uuid, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_2000.csv")
saved = np.load("../data/recipe_embeddings_minilm_2000.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
assert (saved["slugs"] == recipes.slug.to_numpy()).all(), "vectors and CSV rows are out of step"
TITLE = dict(zip(recipes.slug, recipes.title))
print("recipes:", recipes.shape, "| vectors:", doc_emb.shape, "| model:", MODEL)

recipes: (2000, 6) | vectors: (2000, 384) | model: all-MiniLM-L6-v2


The same `uuid5(slug)` ids as lesson 15, loaded with `upload_points`, and the count checked
against the file.

In [2]:
RECIPE_NS = uuid.uuid5(uuid.NAMESPACE_URL, "https://example.com/recipes/")

def recipe_id(slug):
    return str(uuid.uuid5(RECIPE_NS, slug))

client = QdrantClient(":memory:")
client.create_collection("recipes", vectors_config=models.VectorParams(
    size=doc_emb.shape[1], distance=models.Distance.COSINE))
client.upload_points("recipes", [
    models.PointStruct(id=recipe_id(r.slug), vector=doc_emb[i].tolist(),
                       payload={"slug": r.slug, "title": r.title})
    for i, r in enumerate(recipes.itertuples())])
print("points in 'recipes':", client.count("recipes").count)

points in 'recipes': 2000


### 2. Relevance labels, written by hand
A label says how well one recipe answers one query. We use three grades:

- **2** — exactly what was asked for;
- **1** — acceptable, a close substitute;
- **0** — not relevant (every recipe not listed).

The first ten queries are about the 50 original recipes, so we name the right answers one by one.
Most are worded the way a person asks, not the way the recipe is written.

In [3]:
qrels = {
    "a chocolate dessert": {"chocolate-brownies": 2, "chocolate-chip-cookies": 2, "tiramisu": 1},
    "cold soup for a hot day": {"gazpacho": 2},
    "something to dip tortilla chips in": {"guacamole": 2, "mango-salsa": 2, "hummus": 1},
    "a quick vegetarian breakfast": {"pancakes": 2, "avocado-toast": 2, "overnight-oats": 2,
                                     "shakshuka": 1},
    "mussels in white wine": {"moules-marinieres": 2},
    "eggs cooked in a spicy tomato sauce": {"shakshuka": 2},
    "a frozen treat for summer": {"lemon-sorbet": 2, "vanilla-ice-cream": 2},
    "Thai prawn noodles": {"shrimp-pad-thai": 2},
    "a fried Indian pastry snack": {"vegetable-samosas": 2},
    "pizza night": {"margherita-pizza": 2},
}
print(len(qrels), "queries labelled so far")

10 queries labelled so far


The other ten are about the 1,950 generated recipes (lesson 14). Their titles always read
"<flavour> <main> <dish>", for example "Smoky lamb skewers", so a title pattern names every recipe
of a kind exactly. `titles_like` is a labelling aid: it lists the matching slugs so we don't type
15 of them by hand. The grade is still our decision.

In [4]:
FLAVOUR = "(spicy|smoky|lemon garlic|creamy|honey ginger|herby) "

def titles_like(pattern, grade):
    return {s: grade for s, t in TITLE.items() if re.fullmatch(pattern, t.lower())}

qrels |= {
    "a slow-cooked beef stew": titles_like("(.* )?beef stew", 2) | {"chili-con-carne": 1},
    "Indian cottage cheese curry": titles_like(FLAVOUR + "paneer curry", 2)
                                   | titles_like(FLAVOUR + "paneer green curry", 1),
    "lamb skewers": titles_like(FLAVOUR + "lamb skewers", 2),
    "a veggie burger made with beans": titles_like(FLAVOUR + "black bean burger", 2)
                                       | titles_like(FLAVOUR + "chickpea burger", 1),
    "honey ginger duck stir-fry": titles_like(FLAVOUR + "duck stir-fry", 1)
                                  | {"honey-ginger-duck-stir-fry": 2},
    "chickpea wrap": titles_like(FLAVOUR + "chickpea wrap", 2) | {"falafel-wrap": 1},
    "a fish traybake": titles_like(FLAVOUR + "(cod|salmon|tuna) traybake", 2),
    "grilled cheese on skewers": titles_like(FLAVOUR + "halloumi skewers", 2)
                                 | titles_like(FLAVOUR + "paneer skewers", 1),
    "salmon noodle bowl": titles_like(FLAVOUR + "salmon noodle bowl", 2),
    "tofu tacos": titles_like(FLAVOUR + "tofu tacos", 2),
}
print(len(qrels), "queries labelled")

20 queries labelled


Check the key before trusting it. A typo in a slug would silently count as "never found", so
assert that every labelled slug exists, and print how many good answers each query has.

In [5]:
unknown = [s for labels in qrels.values() for s in labels if s not in TITLE]
assert not unknown, f"labels name recipes that do not exist: {unknown}"
for q, labels in qrels.items():
    grades = list(labels.values())
    print(f"{q:<36} grade 2: {grades.count(2):>2}   grade 1: {grades.count(1):>2}")

a chocolate dessert                  grade 2:  2   grade 1:  1
cold soup for a hot day              grade 2:  1   grade 1:  0
something to dip tortilla chips in   grade 2:  2   grade 1:  1
a quick vegetarian breakfast         grade 2:  3   grade 1:  1
mussels in white wine                grade 2:  1   grade 1:  0
eggs cooked in a spicy tomato sauce  grade 2:  1   grade 1:  0
a frozen treat for summer            grade 2:  2   grade 1:  0
Thai prawn noodles                   grade 2:  1   grade 1:  0
a fried Indian pastry snack          grade 2:  1   grade 1:  0
pizza night                          grade 2:  1   grade 1:  0
a slow-cooked beef stew              grade 2:  7   grade 1:  1
Indian cottage cheese curry          grade 2:  4   grade 1:  5
lamb skewers                         grade 2:  6   grade 1:  0
a veggie burger made with beans      grade 2:  6   grade 1:  6
honey ginger duck stir-fry           grade 2:  1   grade 1:  3
chickpea wrap                        grade 2:  6   grad

The ten original-recipe queries have one to four good answers; the generated-recipe queries have 4
to 15, and the "fish traybake" has 15. Keep that spread in mind:
it decides what each metric can reach. Save the key to a file, so later lessons (chunking,
model choice, hybrid search, reranking) sit the same exam.

In [6]:
with open("../data/eval_queries_20.json", "w") as f:
    json.dump({"model": MODEL, "qrels": qrels}, f, indent=1)
print("saved", len(qrels), "labelled queries to ../data/eval_queries_20.json")

saved 20 labelled queries to ../data/eval_queries_20.json


### 3. Run the search for every query
Every query goes through the same model as the recipes. The bank below holds the 20 test
queries plus two used in Try it. The download cell embeds them and writes the cache; the course
ships the file it writes, so skipping it changes nothing.

In [7]:
model = None
EXTRA_QUERIES = ["a fruity dessert", "creamy mushroom pasta"]
EVAL_QUERIES = list(qrels) + EXTRA_QUERIES
CACHE_FILE = "../data/query_embeddings_minilm_eval.npz"
print(len(EVAL_QUERIES), "queries in the bank")

22 queries in the bank


The model cell (skipped without `TUT_ENABLE=download`):

In [8]:
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL, device="cpu")
np.savez(CACHE_FILE, model=MODEL, queries=EVAL_QUERIES, query_emb=model.encode_query(EVAL_QUERIES))
print("saved", len(EVAL_QUERIES), "query embeddings from", MODEL)

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


`embed_query` is lesson 15's: cache first, then the model if loaded, otherwise a clear `KeyError`.

In [9]:
cache = np.load(CACHE_FILE)
assert str(cache["model"]) == MODEL, f"query cache from {cache['model']}, recipes from {MODEL}"
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"], strict=True))

def embed_query(text):
    if text in query_cache:
        return query_cache[text].tolist()
    if model is not None:
        return model.encode_query(text).tolist()
    raise KeyError(f"{text!r} is not in the saved query cache and the model is not loaded")

print("cached:", len(query_cache), "| vector length:", len(embed_query("lamb skewers")))

cached: 22 | vector length: 384


An evaluation sends many queries at once, so use `query_batch_points`: one call, one
`QueryRequest` per query, one `QueryResponse` back per query, in the same order. A **run** is the
result: for each query, the ranked list of slugs. We keep 10 per query.

In [10]:
def vector_run(queries, k=10):
    requests = [models.QueryRequest(query=embed_query(q), limit=k, with_payload=["slug"])
                for q in queries]
    responses = client.query_batch_points("recipes", requests=requests)
    return {q: [p.payload["slug"] for p in r.points] for q, r in zip(queries, responses)}

run = vector_run(list(qrels))
print(len(run), "queries,", len(run["lamb skewers"]), "hits each")
print("lamb skewers ->", run["lamb skewers"][:3])

20 queries, 10 hits each
lamb skewers -> ['creamy-lamb-skewers', 'spicy-lamb-skewers', 'smoky-lamb-skewers']


`show` prints one query's page next to its labels: the grade of each hit, or `-` for 0.

In [11]:
def show(query, ranked, k=5):
    labels = qrels[query]
    print(f"{query!r}  ({len(labels)} labelled)")
    for rank, slug in enumerate(ranked[:k], 1):
        print(f"  {rank}. [{labels.get(slug, '-')}] {TITLE[slug]}")

show("a chocolate dessert", run["a chocolate dessert"])

'a chocolate dessert'  (3 labelled)
  1. [2] Chocolate brownies
  2. [-] Honey ginger paneer pie
  3. [2] Chocolate chip cookies
  4. [-] Strawberry cheesecake
  5. [-] Vanilla ice cream


### 4. Precision@k: how much of the page is useful
**precision@k** = (relevant hits in the top k) / k. A hit counts as relevant at grade 1 or 2.
Above, 2 of the top 5 chocolate hits are labelled, so precision@5 = 2 / 5 = 0.4. Brownies (rank 1)
and cookies (rank 3) are labelled. The paneer pie at rank 2 is a plain miss; cheesecake and ice
cream are desserts, but not chocolate.

In [12]:
def precision_at_k(ranked, labels, k):
    return sum(slug in labels for slug in ranked[:k]) / k

print("precision@5 =", precision_at_k(run["a chocolate dessert"], qrels["a chocolate dessert"], 5))
print("precision@1 =", precision_at_k(run["a chocolate dessert"], qrels["a chocolate dessert"], 1))

precision@5 = 0.4
precision@1 = 1.0


### 5. Recall@k: how much of the good stuff was found
**recall@k** = (relevant hits in the top k) / (number of relevant recipes). The chocolate query
has three labelled recipes and found two of them in the top 5, so recall@5 = 2 / 3.

This is not lesson 13's recall. There, recall compared an approximate index with exact search
("did HNSW find the true nearest neighbours?"). Here it compares the results with what a *person*
says is right. A search can have perfect index recall and still be poor here.

In [13]:
def recall_at_k(ranked, labels, k):
    return sum(slug in labels for slug in ranked[:k]) / len(labels)

print("recall@5  =", round(recall_at_k(run["a chocolate dessert"], qrels["a chocolate dessert"], 5), 3))
print("recall@10 =", round(recall_at_k(run["a chocolate dessert"], qrels["a chocolate dessert"], 10), 3))

recall@5  = 0.667
recall@10 = 0.667


Both numbers have a ceiling set by the labels, not by the search. With one right answer,
precision@5 can never pass 1/5. With 15 right answers, recall@5 can never pass 5/15. Print the
best possible score per query before reading any result, or a perfect search will look broken:

In [14]:
for q in ["pizza night", "a chocolate dessert", "a veggie burger made with beans", "a fish traybake"]:
    n = len(qrels[q])
    print(f"{q:<34} relevant: {n:>2}   best precision@5: {min(n, 5) / 5:.2f}"
          f"   best recall@5: {min(n, 5) / n:.2f}")

pizza night                        relevant:  1   best precision@5: 0.20   best recall@5: 1.00


a chocolate dessert                relevant:  3   best precision@5: 0.60   best recall@5: 1.00
a veggie burger made with beans    relevant: 12   best precision@5: 1.00   best recall@5: 0.42
a fish traybake                    relevant: 15   best precision@5: 1.00   best recall@5: 0.33


### 6. Reciprocal rank and MRR: how soon the first good answer appears
Someone who wants one recipe cares about where the *first* good one sits. **Reciprocal rank** is
1 / (rank of the first relevant hit): 1.0 at the top, 0.5 second, 0.33 third, 0 if none is found
in the list. **MRR** (mean reciprocal rank) is its average over all queries.

In [15]:
def reciprocal_rank(ranked, labels):
    for rank, slug in enumerate(ranked, 1):
        if slug in labels:
            return 1 / rank
    return 0.0

for q in ["a chocolate dessert", "Indian cottage cheese curry", "cold soup for a hot day"]:
    print(f"{q:<30} RR = {reciprocal_rank(run[q], qrels[q]):.2f}")

a chocolate dessert            RR = 1.00


Indian cottage cheese curry    RR = 0.33
cold soup for a hot day        RR = 0.00


The chocolate query scores 1.0 (brownies first). The cottage cheese curry scores 0.33: its first
labelled hit is third. The cold soup scores 0: Gazpacho is nowhere in the top 10. Look at what
came instead:

In [16]:
# two plain-language queries the vector search found hard
show("Indian cottage cheese curry", run["Indian cottage cheese curry"], k=3)
show("cold soup for a hot day", run["cold soup for a hot day"], k=3)

'Indian cottage cheese curry'  (9 labelled)
  1. [-] Lemon garlic sweet potato green curry
  2. [-] Herby sweet potato green curry
  3. [1] Lemon garlic paneer green curry
'cold soup for a hot day'  (1 labelled)
  1. [-] Chicken noodle soup
  2. [-] Herby tempeh soup
  3. [-] Spicy pork soup


For the curry, the first two hits are sweet potato green curries, and the third is a paneer
*green* curry (grade 1). For the soup, the model found "soup" and ignored "cold": among 2,000
recipes, dozens of warm soups crowd out the one chilled soup.

### 7. nDCG: are the best answers first?
Precision, recall and RR treat grade 1 and grade 2 alike. **nDCG@k** uses the grades:

- **DCG@k** adds up each hit's grade divided by log2(rank + 1), so rank 1 counts in full,
  rank 2 by 0.63, rank 5 by 0.39;
- the **ideal DCG** is the same sum for the best possible order of the labels;
- **nDCG** = DCG / ideal DCG, from 0 to 1.

In [17]:
def ndcg_at_k(ranked, labels, k):
    dcg = sum(labels.get(slug, 0) / math.log2(rank + 1) for rank, slug in enumerate(ranked[:k], 1))
    best = sorted(labels.values(), reverse=True)[:k]
    ideal = sum(grade / math.log2(rank + 1) for rank, grade in enumerate(best, 1))
    return dcg / ideal

show("grilled cheese on skewers", run["grilled cheese on skewers"])
q = "grilled cheese on skewers"
print(f"precision@5 = {precision_at_k(run[q], qrels[q], 5):.2f}   nDCG@5 = {ndcg_at_k(run[q], qrels[q], 5):.2f}")

'grilled cheese on skewers'  (11 labelled)
  1. [1] Lemon garlic paneer skewers
  2. [1] Creamy paneer skewers
  3. [1] Spicy paneer skewers
  4. [1] Honey ginger paneer skewers
  5. [-] Spicy lamb skewers
precision@5 = 0.80   nDCG@5 = 0.43


Four of five hits are labelled, so precision@5 is a healthy 0.80. But all four are paneer skewers
(grade 1); the five halloumi skewers (grade 2), the real grilled cheese, are missing. nDCG@5 drops
to 0.43 and tells you so. This is the "linear gain" form of nDCG; some libraries use 2^grade - 1,
which punishes the same mistake harder.

### 8. One report for all 20 queries
Put the four metrics in one function that returns a table: one row per query, then the mean.

In [18]:
def evaluate(run, qrels, k=5):
    rows = {q: {f"P@{k}": precision_at_k(run[q], labels, k),
                f"R@{k}": recall_at_k(run[q], labels, k),
                "RR": reciprocal_rank(run[q], labels),
                f"nDCG@{k}": ndcg_at_k(run[q], labels, k)}
            for q, labels in qrels.items()}
    return pd.DataFrame.from_dict(rows, orient="index")

vector_report = evaluate(run, qrels)
print(vector_report.round(2).to_string())
print("\nmean:", vector_report.mean().round(3).to_dict())

                                     P@5   R@5    RR  nDCG@5
a chocolate dessert                  0.4  0.67  1.00    0.80
cold soup for a hot day              0.0  0.00  0.00    0.00
something to dip tortilla chips in   0.0  0.00  0.00    0.00
a quick vegetarian breakfast         0.0  0.00  0.00    0.00
mussels in white wine                0.2  1.00  1.00    1.00
eggs cooked in a spicy tomato sauce  0.0  0.00  0.00    0.00
a frozen treat for summer            0.4  1.00  1.00    1.00
Thai prawn noodles                   0.2  1.00  1.00    1.00
a fried Indian pastry snack          0.2  1.00  1.00    1.00
pizza night                          0.2  1.00  1.00    1.00
a slow-cooked beef stew              1.0  0.62  1.00    1.00
Indian cottage cheese curry          0.2  0.11  0.33    0.09
lamb skewers                         1.0  0.83  1.00    1.00
a veggie burger made with beans      1.0  0.42  1.00    1.00
honey ginger duck stir-fry           0.2  0.25  1.00    0.56
chickpea wrap           



mean: {'P@5': 0.48, 'R@5': 0.556, 'RR': 0.767, 'nDCG@5': 0.687}


Read the mean row first, then the rows. Mean precision@5 is 0.48, recall@5 0.56, MRR 0.77
(the mean of the RR column) and nDCG@5 0.69. The rows show where the mean comes from:
the named-dish queries ("lamb skewers", "tofu tacos") are near perfect, and four plain-language
queries score 0 on every metric. Those zeros are the to-do list:

In [19]:
# the five weakest queries: lowest reciprocal rank first, then lowest nDCG@5
worst = vector_report.sort_values(["RR", "nDCG@5"]).head(5)
print(worst.round(2).to_string())

                                     P@5   R@5    RR  nDCG@5
cold soup for a hot day              0.0  0.00  0.00    0.00
something to dip tortilla chips in   0.0  0.00  0.00    0.00
a quick vegetarian breakfast         0.0  0.00  0.00    0.00
eggs cooked in a spicy tomato sauce  0.0  0.00  0.00    0.00
Indian cottage cheese curry          0.2  0.11  0.33    0.09

### 9. Check your numbers with `ranx`
Hand-written metrics are easy to get subtly wrong, so compare them once with a library. `ranx`
takes the labels as `Qrels` and the results as a `Run`, both from dicts. A run needs a score for
each hit; we give rank-based scores (10, 9, 8, ...) so ranx keeps our order exactly.

In [20]:
warnings.filterwarnings("ignore", category=Warning, module="ranx")
from ranx import Qrels, Run, evaluate as ranx_evaluate

def to_ranx_run(run):
    return Run.from_dict({q: {slug: float(len(ranked) - i) for i, slug in enumerate(ranked)}
                          for q, ranked in run.items()})

ranx_scores = ranx_evaluate(Qrels.from_dict(qrels), to_ranx_run(run),
                            ["precision@5", "recall@5", "mrr", "ndcg@5"])
print({m: round(float(v), 3) for m, v in ranx_scores.items()})

{'precision@5': 0.48, 'recall@5': 0.556, 'mrr': 0.767, 'ndcg@5': 0.687}


The four ranx numbers equal our mean row: precision@5, recall@5, MRR and nDCG@5 (ranx's `ndcg`
is the linear-gain form; `ndcg_burges` is the 2^grade - 1 one). From now on either is fine; the
hand-written version shows what each number means, ranx gives dozens more metrics and
significance tests.

### 10. Use the numbers: vector search vs keyword search
A metric is only useful when it decides something. Lesson 01's keyword search (shared words
after removing stop words) is the obvious baseline. Ties are broken by slug, so the original 50
recipes get no head start from being first in the file.

In [21]:
STOP_WORDS = {"a", "an", "and", "the", "for", "of", "on", "in", "with", "until", "great"}

def tokenize(text):
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP_WORDS}

recipe_words = {r.slug: tokenize(f"{r.title} {r.description}") for r in recipes.itertuples()}

def keyword_run(queries, k=10):
    out = {}
    for q in queries:
        scores = {slug: len(tokenize(q) & words) for slug, words in recipe_words.items()}
        ranked = sorted((s for s in scores if scores[s] > 0), key=lambda s: (-scores[s], s))
        out[q] = ranked[:k]
    return out

kw_run = keyword_run(list(qrels))
show("cold soup for a hot day", kw_run["cold soup for a hot day"], k=3)

'cold soup for a hot day'  (1 labelled)
  1. [2] Gazpacho
  2. [-] Creamy beef soup
  3. [-] Creamy black bean soup


Gazpacho's description says "perfect on a hot day", so word overlap puts it first. Now the
whole exam, side by side:

In [22]:
keyword_report = evaluate(kw_run, qrels)
summary = pd.DataFrame({"vector": vector_report.mean(), "keyword": keyword_report.mean()})
print(summary.round(3).to_string())
wins = (vector_report["nDCG@5"] - keyword_report["nDCG@5"]).round(3)
print(f"\nnDCG@5 per query: vector better on {(wins > 0).sum()}, keyword better on "
      f"{(wins < 0).sum()}, tied on {(wins == 0).sum()}")

        vector  keyword
P@5      0.480    0.430
R@5      0.556    0.662
RR       0.767    0.802
nDCG@5   0.687    0.733

nDCG@5 per query: vector better on 5, keyword better on 6, tied on 9


On this exam keyword search wins on most averages: recall@5 0.66 against 0.56, nDCG@5 0.73 against
0.69 (vector search keeps a higher precision@5, 0.48 against 0.43). Without labels you would probably have
bet on vector search. The per-query count (vector better on 5, keyword on 6, 9 ties) says more than the
mean: the two methods fail on *different* queries. Print where they disagree most:

In [23]:
disagree = pd.DataFrame({"vector": vector_report["nDCG@5"], "keyword": keyword_report["nDCG@5"],
                         "diff": wins}).sort_values("diff")
print(disagree.head(4).round(2).to_string())
print("...")
print(disagree.tail(3).round(2).to_string())

                                     vector  keyword  diff
cold soup for a hot day                 0.0     1.00 -1.00
eggs cooked in a spicy tomato sauce     0.0     1.00 -1.00
something to dip tortilla chips in      0.0     0.47 -0.47
a quick vegetarian breakfast            0.0     0.21 -0.21
...
                           vector  keyword  diff
grilled cheese on skewers    0.43     0.00  0.43
a slow-cooked beef stew      1.00     0.53  0.47
a fish traybake              1.00     0.13  0.87


Before reading the table, look closer at "grilled cheese on skewers": how many recipes share two
of its words, and where do the first halloumi skewers land in the full keyword ranking?

In [24]:
q = "grilled cheese on skewers"
full = keyword_run([q], k=2000)[q]
overlap = sum(len(tokenize(q) & recipe_words[s]) == 2 for s in recipe_words)
print("recipes sharing 2 words:", overlap, "| first halloumi skewers rank:",
      next(i for i, s in enumerate(full, 1) if "halloumi-skewers" in s))

recipes sharing 2 words: 119 | first halloumi skewers rank: 26


Keyword search wins when the user's words appear in the recipe ("hot day", "tomato sauce",
"chips"). Vector search wins when the words don't settle it. "fish" never appears in a cod or
tuna traybake. And "grilled cheese on skewers" shares two words ("grilled", "skewers") with 119
recipes, so keyword search is left breaking a 119-way tie alphabetically, and the first halloumi
skewers lands at rank 26 (both numbers are in the line printed above). Neither method is enough alone, which is exactly what hybrid search
(lessons 19-20) fixes, and this exam is how lesson 26 will prove it.

Three cautions. With 20 queries, one query can move a mean by up to 0.05 (1/20), so a gap of a few hundredths is
noise. Labels written by someone reading the recipes share their words, which flatters keyword
search; real test queries come from real users (search logs). And word-overlap scores tie a lot,
and our slug tie-break is arbitrary, so some keyword rows measure alphabetical order, not
relevance. A real keyword ranker (BM25, lesson 19) breaks ties by how rare each word is.

## What just happened
- You wrote an answer key (20 queries, 100 graded pairs), checked every slug exists, and saved it to `data/eval_queries_20.json`.
- `query_batch_points` searched all 20 queries in one call and gave a run: 10 ranked slugs per query.
- Each metric answered a different question: chocolate query P@5 0.4, R@5 0.67, RR 1.0; "grilled cheese on skewers" P@5 0.80 but nDCG@5 0.43 (wrong grade on top).
- The report's mean row (P@5 0.48, R@5 0.56, MRR 0.77, nDCG@5 0.69) matched `ranx` exactly, and the zero rows named the queries to fix.
- Keyword search beat vector search on average, but the two won different queries: the measurement, not intuition, says "combine them".

## Common mistakes

**1. Comparing the wrong ids.** The labels use slugs; Qdrant returns point ids (UUIDs). Build
the run from `p.id` and nothing ever matches. No error, just a search that looks useless:

In [25]:
try:
    responses = client.query_batch_points("recipes", requests=[
        models.QueryRequest(query=embed_query(q), limit=10) for q in qrels])
    id_run = {q: [str(p.id) for p in r.points] for q, r in zip(qrels, responses)}
    id_report = evaluate(id_run, qrels)
    print(id_report.mean().round(3).to_dict())
    assert id_report["RR"].sum() > 0, "every query scored 0: the run's ids never match the labels"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

{'P@5': 0.0, 'R@5': 0.0, 'RR': 0.0, 'nDCG@5': 0.0}
AssertionError: every query scored 0: the run's ids never match the labels


It prints all zeros, then `AssertionError: every query scored 0: the run's ids never match the
labels`. Fix: put the labels' key in the payload and build the run from it (`p.payload["slug"]`,
as `vector_run` does), or label by `recipe_id(slug)`. An all-zero report is almost always this
bug, not a bad search.

**2. A query with no relevant labels.** Adding a query and forgetting its labels (or labelling
it "nothing is relevant") breaks recall: there is nothing to divide by.

In [26]:
try:
    recall_at_k(vector_run(["creamy mushroom pasta"])["creamy mushroom pasta"], {}, 5)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ZeroDivisionError: division by zero


It prints `ZeroDivisionError: division by zero`. Fix: check the key before evaluating, as
step 2 does, and keep "no good answer exists" queries in a separate set. They test something
else (should the finder say "nothing found"? lesson 10's `score_threshold`).

In [27]:
# the fix in action: list queries with no labels before evaluating
empty = [q for q, labels in qrels.items() if not labels]
print("queries without labels:", empty)

queries without labels: []


## Try it
1. **Grow the exam.** Add labels for the two extra queries in the cache, `"a fruity dessert"`
   (for example apple pie and strawberry cheesecake as 2, lemon sorbet as 1) and
   `"creamy mushroom pasta"` (use `titles_like`), then re-run both methods on all 22 queries.
   How much do the means move?
2. **How deep must you look?** Compute mean recall@k for k = 1, 3, 5, 10, 20 for vector and
   keyword search. At which k does vector search find most of the labelled recipes? (Reranking,
   lesson 21, re-orders such a deeper list.)

In [28]:
# your code here

Clean up: close the local client (the `:memory:` collection disappears with it).

In [29]:
# the in-memory collection is gone after this
client.close()
print("client closed")

client closed


## Key terms
- **relevance label** — a judgement of how well one item answers one query, here a grade: 2 exactly right, 1 acceptable, 0 not relevant.
- **qrels** — the full set of relevance labels for a test set: query → {item: grade}; the answer key.
- **run** — a search method's output on a test set: query → ranked list of items.
- **precision@k** — relevant hits in the top k divided by k; how much of the page is useful.
- **recall@k** — relevant hits in the top k divided by the number of relevant items; not the same as ANN recall (lesson 13).
- **reciprocal rank / MRR** — 1 / rank of the first relevant hit (0 if none); MRR is its mean over queries.
- **nDCG@k** — the top k's grades, each divided by log2(rank + 1), summed and divided by the best possible sum; rewards best grades first.
- **baseline** — a simple method (here keyword search) every new method must beat on the same test set.